# QLoRA Fine-Tuning (Colab)

Fine-tunes `mistralai/Mistral-7B-Instruct-v0.3` on the tri-modal scenario dataset using the **fixed** pipeline (`fine_tune.py`). No training logic lives in this notebook — it ships the repo modules and runs the script, so the prompt/precision/masking fixes always apply.

Steps: GPU check -> install -> ship code -> upload `training_data.jsonl` -> train -> download the adapter.

**Precision is auto-selected:** bf16 on A100, fp16 on T4.

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('bf16 supported (A100=True, T4=False):', torch.cuda.is_bf16_supported())
else:
    print('No GPU. Runtime -> Change runtime type -> T4 GPU')

In [ ]:
!pip install -q -r requirements_finetune.txt
print('Installed fine-tuning requirements.')

In [ ]:
# Ship the fixed repo modules into this runtime (source of truth).
# These are the repaired pipeline files.
import pathlib
pathlib.Path("prompt_format.py").write_text("\"\"\"The prompt around every training example and every inference request.\n\nThe shipped adapter (fine_tuning/output) was trained by the original Colab\nnotebook on exactly this text, with TRL adding the <s> and appending the </s>:\n\n    <s>[INST] {instruction}\\\\n\\\\n{input} [/INST]\\\\n{output}</s>\n\nNote the space before [/INST] and the newline after it. Mistral's own chat\ntemplate has neither, so the tokenizer's apply_chat_template must not be used\nfor this model: its prompt is input the adapter never saw.\n\nTraining and every inference path build their text here, so the two cannot\ndrift. The prompt carries its own <s>, so tokenize it with\nadd_special_tokens=False.\n\"\"\"\n\nfrom __future__ import annotations\n\n\nBOS = \"<s>\"\nEOS = \"</s>\"\n\n\ndef user_message(record: dict) -> str:\n    \"\"\"The request itself, for a record with instruction and input. Another model\n    gets this as its user turn, in its own chat template.\"\"\"\n    return f\"{record['instruction']}\\n\\n{record['input']}\"\n\n\ndef build_prompt(record: dict) -> str:\n    \"\"\"Everything before the model's answer.\"\"\"\n    return f\"{BOS}[INST] {user_message(record)} [/INST]\\n\"\n\n\ndef build_completion(record: dict) -> str:\n    \"\"\"The answer the model is trained to produce after build_prompt(record).\"\"\"\n    return f\"{record['output']}{EOS}\"\n", encoding='utf-8')
pathlib.Path("data_split.py").write_text("\"\"\"The train / validation / test split, shared by training and evaluation.\n\nThe test set is the 17 records the shipped adapter never trained on. The\noriginal Colab notebook split the 81 records of dataset/training_data.jsonl with\n`datasets` train_test_split(test_size=0.2, seed=42) and trained on the other 64.\nTheir ids are frozen here rather than recomputed, so the split cannot move out\nfrom under the adapter; tests/test_held_out_split.py recomputes them with\n`datasets` when it is installed.\n\nValidation is carved from the other 64, for a retrain's eval loss and early\nstopping, so model selection never looks at the test set.\n\nThe test records are also written to evaluation/test_inputs.jsonl, which is\ncommitted and shipped to the Colab notebooks.\n\"\"\"\n\nfrom __future__ import annotations\n\nimport json\nimport random\nfrom pathlib import Path\n\n\nDEFAULT_TEST_INPUTS = \"evaluation/test_inputs.jsonl\"\n\nHELD_OUT_SOURCE_IDS = (\n    \"C0011\", \"C0013\", \"C0015\", \"C0017\", \"C0023\", \"C0027\", \"C0037\", \"C0043\",\n    \"C0046\", \"C0049\", \"C0051\", \"C0055\", \"G0010\", \"G0020\", \"G0049\", \"G0119\",\n    \"G1047\",\n)\n\n# The fields kept in test_inputs.jsonl. `output` is the reference answer, used\n# only for comparison and never shown to the model.\nTEST_FIELDS = (\"idx\", \"source_id\", \"instruction\", \"input\", \"output\")\n\n\ndef load_records(jsonl_path: str | Path) -> list[dict]:\n    \"\"\"Every record of a JSONL file, skipping blank lines.\"\"\"\n    with open(jsonl_path, \"r\", encoding=\"utf-8\") as f:\n        return [json.loads(line) for line in f if line.strip()]\n\n\ndef source_id(record: dict) -> str:\n    \"\"\"The record's stable id.\"\"\"\n    return record[\"_meta\"][\"source_id\"]\n\n\ndef split(records: list[dict], val_frac: float = 0.15, seed: int = 42) -> dict:\n    \"\"\"{\"train\", \"val\", \"test\"}, each a list of records with `idx` and `source_id` added.\"\"\"\n    annotated = [dict(record, idx=i, source_id=source_id(record))\n                 for i, record in enumerate(records)]\n\n    test = [r for r in annotated if r[\"source_id\"] in HELD_OUT_SOURCE_IDS]\n    missing = set(HELD_OUT_SOURCE_IDS) - {r[\"source_id\"] for r in test}\n    if missing:\n        raise ValueError(f\"held-out records missing from the dataset: {sorted(missing)}\")\n\n    pool = [r for r in annotated if r[\"source_id\"] not in HELD_OUT_SOURCE_IDS]\n    order = list(range(len(pool)))\n    random.Random(seed).shuffle(order)\n    val_idx = set(order[:max(1, round(val_frac * len(pool)))])\n\n    return {\n        \"train\": [r for i, r in enumerate(pool) if i not in val_idx],\n        \"val\": [r for i, r in enumerate(pool) if i in val_idx],\n        \"test\": test,\n    }\n\n\ndef write_test_inputs(path: str | Path, test_records: list[dict]) -> None:\n    \"\"\"Write the held-out records as JSONL.\"\"\"\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    with path.open(\"w\", encoding=\"utf-8\") as f:\n        for record in test_records:\n            f.write(json.dumps({k: record[k] for k in TEST_FIELDS}, ensure_ascii=False) + \"\\n\")\n\n\ndef read_test_inputs(path: str | Path) -> list[dict]:\n    \"\"\"The held-out records written by write_test_inputs.\n\n    Raises if the file holds a record the adapter was trained on, so a stale or\n    hand-made test file cannot pass training data off as unseen.\n    \"\"\"\n    records = load_records(path)\n    trained_on = [r[\"source_id\"] for r in records if r[\"source_id\"] not in HELD_OUT_SOURCE_IDS]\n    if trained_on:\n        raise ValueError(f\"{path} holds records the adapter was trained on: {trained_on}. \"\n                         \"Regenerate it with: python evaluation/make_test_inputs.py\")\n    return records\n", encoding='utf-8')
pathlib.Path("finetuning.py").write_text("\"\"\"\nfinetuning.py\n-------------\nQLoRA fine-tuning of Mistral-7B-Instruct on the tri-modal cyber attack\nscenario dataset generated by generate_outputs.py.\n\nWhat is QLoRA?\n  Q  = Quantization: loads the model in 4-bit precision (uses ~4-5GB VRAM\n       instead of ~28GB), making it runnable on a free Colab/Kaggle GPU.\n  Lo = Low-Rank: instead of updating all 7 billion weights, we attach\n       small \"adapter\" matrices (LoRA) to key layers and only train those.\n  RA = Rank Adaptation: the adapters have rank r (default 16), meaning each\n       adapter is a pair of tiny matrices (r x hidden_dim) — very few params.\n\nResult: a fine-tuned model that knows the cybersecurity scenario format\nwithout needing a $10,000 GPU.\n\nUsage (run from scenario_generation/):\n    python fine_tuning/fine_tune.py \\\n        --data  dataset/training_data.jsonl \\\n        --out   fine_tuning/output/\n\nOffline probes (no GPU / no ML stack required):\n    python fine_tuning/fine_tune.py --check_data        # split + token lengths\n    python fine_tuning/fine_tune.py --smoke --out /tmp/s # 3-step CUDA-guarded run\n\nRequirements:\n    pip install -r fine_tuning/requirements_finetune.txt\n\nNotes on portability:\n  * The heavy ML imports (torch, transformers, trl, peft, datasets) are done\n    LAZILY inside the functions that need them, so --help, --check_data and the\n    GPU-less --smoke path work on a machine with no ML stack installed.\n  * Written against the versions pinned in requirements_finetune.txt (TRL 0.29,\n    transformers 5), the stack that trained the shipped adapter.\n\"\"\"\n\nimport argparse\nimport math\nimport sys\nfrom pathlib import Path\n\n# Local shared modules (same directory). These are dependency-free so they import\n# fine even without the ML stack.\nimport prompt_format\nimport data_split\n\n\n# ── Constants ─────────────────────────────────────────────────────────────────\n\nBASE_MODEL   = \"mistralai/Mistral-7B-Instruct-v0.3\"\nMAX_SEQ_LEN  = 2048   # max token length per training example (see fix C)\n\n# LoRA target-module presets.\nALL_LINEAR = [\"q_proj\", \"k_proj\", \"v_proj\", \"o_proj\", \"gate_proj\", \"up_proj\", \"down_proj\"]\nATTN_ONLY  = [\"q_proj\", \"k_proj\", \"v_proj\", \"o_proj\"]\n\n# The three section headers the model must emit, in order (used by the self-test).\nREQUIRED_HEADERS = [\"[NATURAL LANGUAGE]\", \"[MATHEMATICAL EXPRESSION]\", \"[ATTACK GRAPH]\"]\n\n\n# ── Dependency check (only invoked on the real-training / GPU paths) ───────────\n\ndef check_imports():\n    \"\"\"Friendly check for the heavy ML stack; called only when we actually train.\"\"\"\n    missing = []\n    for pkg, import_name in [\n        (\"transformers\", \"transformers\"),\n        (\"peft\",         \"peft\"),\n        (\"trl\",          \"trl\"),\n        (\"bitsandbytes\", \"bitsandbytes\"),\n        (\"datasets\",     \"datasets\"),\n        (\"accelerate\",   \"accelerate\"),\n        (\"torch\",        \"torch\"),\n    ]:\n        try:\n            __import__(import_name)\n        except ImportError:\n            missing.append(pkg)\n    if missing:\n        print(\"[!] Missing packages. Run:\")\n        print(f\"    pip install {' '.join(missing)}\")\n        sys.exit(1)\n\n\n# ── Dataset (shared split + the prompt the adapter was trained on) ────────────\n\ndef tokenize_example(tokenizer, record: dict) -> dict:\n    \"\"\"input_ids for prompt + answer, and a completion_mask that is 1 on the answer.\n\n    Tokenized here rather than by TRL so the ids are exactly the ones inference\n    sees: prompt_format embeds the <s>, so no special tokens are added, and the\n    loss only falls on the answer.\n    \"\"\"\n    prompt = prompt_format.build_prompt(record)\n    prompt_ids = tokenizer(prompt, add_special_tokens=False)[\"input_ids\"]\n    input_ids = tokenizer(prompt + prompt_format.build_completion(record),\n                          add_special_tokens=False)[\"input_ids\"]\n    if input_ids[:len(prompt_ids)] != prompt_ids:\n        raise ValueError(f\"{record['_meta']['source_id']}: the prompt does not tokenize \"\n                         \"as a prefix of the full example\")\n    return {\n        \"input_ids\": input_ids,\n        \"completion_mask\": [0] * len(prompt_ids) + [1] * (len(input_ids) - len(prompt_ids)),\n    }\n\n\ndef load_dataset_split(data_path, tokenizer, val_frac, seed):\n    \"\"\"Tokenized train and validation Datasets, plus the split they came from.\n\n    The held-out test records are never trained on; see data_split.py.\n    \"\"\"\n    from datasets import Dataset\n\n    split = data_split.split(data_split.load_records(data_path), val_frac=val_frac, seed=seed)\n    train_ds = Dataset.from_list([tokenize_example(tokenizer, r) for r in split[\"train\"]])\n    val_ds = Dataset.from_list([tokenize_example(tokenizer, r) for r in split[\"val\"]])\n    print(f\"[+] Split: train {len(train_ds)} / val {len(val_ds)} / \"\n          f\"test {len(split['test'])} (held out, never trained on)\")\n    return train_ds, val_ds, split\n\n\n# ── Precision / hardware autodetect (fix D1) ──────────────────────────────────\n\ndef detect_bf16() -> bool:\n    \"\"\"True on GPUs that support bfloat16 (e.g. A100); False on T4 (fp16 only).\"\"\"\n    import torch\n    return torch.cuda.is_available() and torch.cuda.is_bf16_supported()\n\n\n# ── Model + tokenizer loading (true QLoRA, fix D) ─────────────────────────────\n\ndef load_model_and_tokenizer(model_name: str, trust_remote_code: bool):\n    \"\"\"\n    Load Mistral-7B in 4-bit quantized form (true QLoRA setup).\n\n    BitsAndBytesConfig:\n      - load_in_4bit              : compress base weights to 4-bit (~4-5GB VRAM)\n      - bnb_4bit_quant_type=\"nf4\" : NormalFloat4, best quality for 4-bit\n      - bnb_4bit_use_double_quant : quantize the quantization constants too\n      - bnb_4bit_compute_dtype    : bf16 on A100, fp16 on T4 — MATCHED to the\n                                    mixed-precision used by the trainer (fix D1).\n    We do NOT set bnb_4bit_quant_storage (default uint8) and never use fp32\n    compute (fixes D2). After loading we run prepare_model_for_kbit_training so\n    LoRA trains correctly on the quantized base (fix D4).\n    \"\"\"\n    import torch\n    from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig\n    from peft import prepare_model_for_kbit_training\n\n    bf16 = detect_bf16()\n    compute_dtype = torch.bfloat16 if bf16 else torch.float16\n    print(f\"[+] Precision: {'bf16' if bf16 else 'fp16'} \"\n          f\"(compute_dtype={compute_dtype})\")\n\n    print(f\"[+] Loading base model: {model_name}\")\n    print(\"    This downloads ~14GB on first run. Subsequent runs use cache.\")\n\n    bnb_config = BitsAndBytesConfig(\n        load_in_4bit=True,\n        bnb_4bit_quant_type=\"nf4\",\n        bnb_4bit_use_double_quant=True,\n        bnb_4bit_compute_dtype=compute_dtype,\n    )\n\n    model = AutoModelForCausalLM.from_pretrained(\n        model_name,\n        quantization_config=bnb_config,\n        device_map=\"auto\",                 # places layers on GPU automatically\n        trust_remote_code=trust_remote_code,\n    )\n\n    model.config.use_cache = False         # required for gradient checkpointing\n    model.config.pretraining_tp = 1\n\n    tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=trust_remote_code)\n    # Mistral has no pad token by default; reuse EOS so batches can pad.\n    if tokenizer.pad_token is None:\n        tokenizer.pad_token = tokenizer.eos_token\n    tokenizer.padding_side = \"right\"       # right padding for causal-LM training\n\n    # k-bit prep: upcasts layernorms to fp32, enables input grads, and turns on\n    # gradient checkpointing — must run AFTER the 4-bit load and BEFORE LoRA.\n    model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)\n\n    print(\"[+] Model loaded and prepared for k-bit training\")\n    return model, tokenizer, bf16\n\n\n# ── LoRA config (fix G2: target modules configurable) ─────────────────────────\n\ndef attach_lora(model, r, alpha, dropout, target_modules):\n    \"\"\"\n    Attach LoRA adapter matrices. Only A/B adapter weights train; the 4-bit base\n    is frozen. By default we adapt ALL linear projections (attention + MLP), which\n    generally fits a new output format better than attention-only.\n    \"\"\"\n    from peft import LoraConfig, get_peft_model, TaskType\n\n    lora_config = LoraConfig(\n        r=r,\n        lora_alpha=alpha,\n        target_modules=target_modules,\n        lora_dropout=dropout,\n        bias=\"none\",\n        task_type=TaskType.CAUSAL_LM,\n    )\n    model = get_peft_model(model, lora_config)\n\n    trainable, total = model.get_nb_trainable_parameters()\n    print(\"[+] LoRA adapters attached\")\n    print(f\"    Target modules   : {target_modules}\")\n    print(f\"    Trainable params : {trainable:,}  ({100 * trainable / total:.2f}% of total)\")\n    print(f\"    Total params     : {total:,}\")\n    return model\n\n\n# ── SFTConfig (TRL 0.29, as pinned in requirements_finetune.txt) ─────────────\n\ndef build_sft_config(output_dir, args, bf16, eval_mode=\"epoch\", max_steps=None):\n    \"\"\"The training config. eval_mode: \"epoch\" (full run), \"steps\" (smoke) or \"no\".\"\"\"\n    from trl import SFTConfig\n\n    kwargs = dict(\n        output_dir=output_dir,\n        per_device_train_batch_size=args.batch_size,\n        gradient_accumulation_steps=args.grad_accum,\n        optim=\"paged_adamw_8bit\",\n        learning_rate=args.lr,\n        lr_scheduler_type=\"cosine\",\n        warmup_steps=5,\n        bf16=bf16,\n        fp16=not bf16,\n        logging_steps=2 if max_steps else 10,\n        save_total_limit=2,\n        report_to=\"none\",\n        max_length=args.max_seq_len,\n        completion_only_loss=True,\n        packing=False,\n        max_grad_norm=0.3,\n        gradient_checkpointing=True,\n        gradient_checkpointing_kwargs={\"use_reentrant\": False},\n        seed=args.seed,\n        data_seed=args.seed,\n    )\n    if max_steps is not None:\n        kwargs[\"max_steps\"] = max_steps\n    else:\n        kwargs[\"num_train_epochs\"] = args.epochs\n\n    if eval_mode == \"no\":\n        kwargs.update(eval_strategy=\"no\", save_strategy=\"no\")\n    else:\n        if eval_mode == \"steps\":\n            kwargs.update(eval_strategy=\"steps\", eval_steps=2, save_strategy=\"steps\", save_steps=2)\n        else:\n            kwargs.update(eval_strategy=\"epoch\", save_strategy=\"epoch\")\n        kwargs.update(per_device_eval_batch_size=1, load_best_model_at_end=True,\n                      metric_for_best_model=\"eval_loss\", greater_is_better=False)\n\n    return SFTConfig(**kwargs)\n\n\n# ── Trainer ───────────────────────────────────────────────────────────────────\n\ndef build_trainer(model, tokenizer, sft_config, train_ds, eval_ds):\n    from trl import SFTTrainer\n    from transformers import EarlyStoppingCallback\n\n    kwargs = dict(model=model, args=sft_config, train_dataset=train_ds,\n                  processing_class=tokenizer)\n    if eval_ds is not None:\n        kwargs[\"eval_dataset\"] = eval_ds\n        kwargs[\"callbacks\"] = [EarlyStoppingCallback(early_stopping_patience=2)]\n    return SFTTrainer(**kwargs)\n\n\n# ── Post-training self-test (fix P3) ──────────────────────────────────────────\n\ndef post_training_selftest(model, tokenizer, test_rec, max_new_tokens):\n    \"\"\"\n    Generate one held-out sample and assert (i) all three headers present in\n    order and (ii) EOS reached before max_new_tokens. Prints PASS/FAIL.\n    \"\"\"\n    import torch\n\n    model.config.use_cache = True\n    if hasattr(model, \"gradient_checkpointing_disable\"):\n        model.gradient_checkpointing_disable()\n    model.eval()\n\n    prompt = prompt_format.build_prompt(test_rec)\n    inputs = tokenizer(prompt, return_tensors=\"pt\", add_special_tokens=False).to(model.device)\n    n = inputs[\"input_ids\"].shape[1]\n    with torch.no_grad():\n        out = model.generate(\n            **inputs,\n            max_new_tokens=max_new_tokens,\n            do_sample=False,\n            eos_token_id=tokenizer.eos_token_id,\n            pad_token_id=tokenizer.pad_token_id,\n        )\n    new_ids = out[0][n:].tolist()\n    text = tokenizer.decode(new_ids, skip_special_tokens=True)\n\n    positions = [text.find(h) for h in REQUIRED_HEADERS]\n    headers_ok = all(p >= 0 for p in positions) and positions == sorted(positions)\n    eos_hit = tokenizer.eos_token_id in new_ids\n\n    print()\n    print(\"─\" * 60)\n    print(\"[self-test] held-out generation\")\n    print(f\"    headers present & in order : {headers_ok}  ({positions})\")\n    print(f\"    EOS reached before cap     : {eos_hit}  \"\n          f\"(generated {len(new_ids)}/{max_new_tokens} tokens)\")\n    ok = headers_ok and eos_hit\n    print(f\"    RESULT: {'PASS' if ok else 'FAIL'}\")\n    print(\"─\" * 60)\n    return ok\n\n\n# ── Optional adapter merge (fix P4) ───────────────────────────────────────────\n\ndef merge_adapter(base_model, adapter_dir, out_dir, trust_remote_code):\n    \"\"\"\n    Merge the LoRA adapter into a fresh fp16 copy of the base model and save it\n    alongside the adapter. NOTE: this loads the FULL (non-quantized) base in\n    fp16, so it needs ~14-16GB of RAM/VRAM — far more than 4-bit training.\n    \"\"\"\n    import torch\n    from transformers import AutoModelForCausalLM, AutoTokenizer\n    from peft import PeftModel\n\n    print(\"[+] --merge: loading base in fp16 to merge the adapter \"\n          \"(needs ~14-16GB; off by default)…\")\n    base = AutoModelForCausalLM.from_pretrained(\n        base_model, torch_dtype=torch.float16, device_map=\"auto\",\n        trust_remote_code=trust_remote_code,\n    )\n    merged = PeftModel.from_pretrained(base, adapter_dir)\n    merged = merged.merge_and_unload()\n    mdir = Path(out_dir) / \"merged\"\n    merged.save_pretrained(mdir, safe_serialization=True)\n    AutoTokenizer.from_pretrained(base_model, trust_remote_code=trust_remote_code).save_pretrained(mdir)\n    print(f\"[✓] Merged fp16 model saved to: {mdir}\")\n\n\n# ── Training orchestration ────────────────────────────────────────────────────\n\ndef masked_prompt_tokens(example) -> int:\n    \"\"\"How many leading tokens of a tokenized example carry no loss.\"\"\"\n    return example[\"completion_mask\"].index(1)\n\n\ndef train(args):\n    \"\"\"Run the supervised fine-tuning loop, with loss on the answer only.\"\"\"\n    check_imports()\n    from transformers import set_seed\n    set_seed(args.seed)\n\n    model, tokenizer, bf16 = load_model_and_tokenizer(args.base, args.trust_remote_code)\n    model = attach_lora(model, args.r, args.alpha, args.lora_dropout, args.target_modules)\n\n    train_ds, val_ds, split = load_dataset_split(args.data, tokenizer, args.val_frac, args.seed)\n    too_long = [len(ids) for ids in train_ds[\"input_ids\"] + val_ds[\"input_ids\"]\n                if len(ids) > args.max_seq_len]\n    if too_long:\n        print(f\"[!] {len(too_long)} examples exceed max_length={args.max_seq_len} \"\n              f\"(longest {max(too_long)}). Truncating them would cut off the end of the \"\n              \"answer and its </s>; raise --max_seq_len.\")\n        sys.exit(1)\n\n    eff_batch = args.batch_size * args.grad_accum\n    steps_per_epoch = math.ceil(len(train_ds) / eff_batch)\n    print()\n    print(\"[+] Starting training…\")\n    print(f\"    Output dir       : {args.out}\")\n    print(f\"    Epochs           : {args.epochs}\")\n    print(f\"    Effective batch  : {eff_batch} (bs {args.batch_size} x accum {args.grad_accum})\")\n    print(f\"    Optimizer steps  : ~{steps_per_epoch * args.epochs} \"\n          f\"({steps_per_epoch}/epoch x {args.epochs})\")\n    print(f\"    Prompt tokens without loss (first example): {masked_prompt_tokens(train_ds[0])}\")\n    print()\n\n    trainer = build_trainer(model, tokenizer, build_sft_config(args.out, args, bf16),\n                            train_ds, val_ds)\n    trainer.train()\n    print(f\"[+] Best checkpoint: {trainer.state.best_model_checkpoint}  \"\n          f\"(eval_loss={trainer.state.best_metric})\")\n\n    print(\"[+] Saving final adapter weights…\")\n    trainer.save_model(args.out)\n    tokenizer.save_pretrained(args.out)\n    print(f\"[✓] Model saved to: {args.out}\")\n\n    ok = post_training_selftest(model, tokenizer, split[\"test\"][0], args.max_new_tokens)\n\n    if args.merge:\n        merge_adapter(args.base, args.out, args.out, args.trust_remote_code)\n\n    print()\n    print(\"Next step: run evaluation/judge_eval.py to score the fine-tuned model\")\n    if not ok:\n        sys.exit(1)\n\n\n# ── --check_data probe (no model load) ────────────────────────────────────────\n\ndef run_check_data(args) -> int:\n    print(\"=\" * 60)\n    print(\"  --check_data : data, split and token-length probe (no model load)\")\n    print(\"=\" * 60)\n\n    if not Path(args.data).exists():\n        print(f\"[!] Dataset not found: {args.data}\")\n        return 1\n\n    split = data_split.split(data_split.load_records(args.data),\n                             val_frac=args.val_frac, seed=args.seed)\n    print(f\"[+] Split: train {len(split['train'])} / val {len(split['val'])} / \"\n          f\"test {len(split['test'])}\")\n\n    try:\n        from transformers import AutoTokenizer\n        tokenizer = AutoTokenizer.from_pretrained(args.base, trust_remote_code=args.trust_remote_code)\n    except Exception as e:\n        print(f\"[i] Tokenizer unavailable ({type(e).__name__}); token checks skipped.\")\n        print(\"\\n[✓] check_data: split OK\")\n        return 0\n\n    examples = [tokenize_example(tokenizer, r) for r in split[\"train\"] + split[\"val\"]]\n    lengths = sorted(len(e[\"input_ids\"]) for e in examples)\n    print(f\"[+] Training example length in tokens: min={lengths[0]} \"\n          f\"median={lengths[len(lengths) // 2]} max={lengths[-1]}\")\n    print(f\"[+] Prompt tokens without loss (first example): {masked_prompt_tokens(examples[0])}\")\n\n    over = sum(1 for n in lengths if n > args.max_seq_len)\n    if over:\n        print(f\"[!] {over} example(s) exceed max_length={args.max_seq_len}. \"\n              \"Raise --max_seq_len before training.\")\n        return 1\n\n    print(\"\\n[✓] check_data: ALL CHECKS PASSED\")\n    return 0\n\n\n# ── --smoke probe (CUDA-guarded) ──────────────────────────────────────────────\n\ndef run_smoke(args) -> int:\n    print(\"=\" * 60)\n    print(\"  --smoke : 3-step end-to-end training + 1 generation\")\n    print(\"=\" * 60)\n    try:\n        import torch\n    except ImportError:\n        torch = None\n    if torch is None or not torch.cuda.is_available():\n        print(\"[smoke] No CUDA GPU here, so smoke is a NO-OP.\")\n        print(\"        On Colab with a GPU this runs max_steps=3 training + one\")\n        print(\"        generation to exercise the whole path fast.\")\n        return 0\n\n    check_imports()\n    from transformers import set_seed\n    set_seed(args.seed)\n\n    model, tokenizer, bf16 = load_model_and_tokenizer(args.base, args.trust_remote_code)\n    model = attach_lora(model, args.r, args.alpha, args.lora_dropout, args.target_modules)\n    train_ds, val_ds, split = load_dataset_split(args.data, tokenizer, args.val_frac, args.seed)\n\n    sft_config = build_sft_config(args.out, args, bf16, eval_mode=\"steps\", max_steps=3)\n    trainer = build_trainer(model, tokenizer, sft_config, train_ds, val_ds)\n    trainer.train()\n    print(f\"[smoke] best checkpoint: {trainer.state.best_model_checkpoint} \"\n          f\"(eval_loss={trainer.state.best_metric})\")\n\n    post_training_selftest(model, tokenizer, split[\"test\"][0],\n                           max_new_tokens=min(args.max_new_tokens, 256))\n    print(\"[✓] smoke complete\")\n    return 0\n\n\n# ── target-module parsing (fix G2) ────────────────────────────────────────────\n\ndef parse_target_modules(value: str):\n    if value == \"all\":\n        return ALL_LINEAR\n    if value == \"attn\":\n        return ATTN_ONLY\n    return [m.strip() for m in value.split(\",\") if m.strip()]\n\n\n# ── Main ──────────────────────────────────────────────────────────────────────\n\ndef main():\n    parser = argparse.ArgumentParser(description=\"QLoRA fine-tuning for scenario generation\")\n    parser.add_argument(\"--data\", default=\"dataset/training_data.jsonl\",\n                        help=\"Path to training_data.jsonl\")\n    parser.add_argument(\"--out\", default=\"fine_tuning/output\",\n                        help=\"Output directory for adapter weights\")\n    parser.add_argument(\"--base\", default=BASE_MODEL, help=f\"Base model (default: {BASE_MODEL})\")\n\n    # Hyperparameters (fix G5) — current values as defaults.\n    parser.add_argument(\"--epochs\", type=int, default=3)\n    parser.add_argument(\"--lr\", type=float, default=2e-4)\n    parser.add_argument(\"--r\", type=int, default=16, help=\"LoRA rank\")\n    parser.add_argument(\"--alpha\", type=int, default=32, help=\"LoRA alpha\")\n    parser.add_argument(\"--lora_dropout\", type=float, default=0.05)\n    parser.add_argument(\"--max_seq_len\", type=int, default=MAX_SEQ_LEN)\n    parser.add_argument(\"--target_modules\", default=\"all\",\n                        help='\"all\" (default, all linear), \"attn\" (attention-only), '\n                             \"or a comma list e.g. q_proj,v_proj\")\n    parser.add_argument(\"--batch_size\", type=int, default=1)\n    parser.add_argument(\"--grad_accum\", type=int, default=4)\n\n    # Split / reproducibility. The test set is fixed; see data_split.py.\n    parser.add_argument(\"--val_frac\", type=float, default=0.15,\n                        help=\"Validation fraction of the train pool\")\n    parser.add_argument(\"--seed\", type=int, default=42)\n\n    # Behaviour flags.\n    parser.add_argument(\"--trust_remote_code\", action=\"store_true\",\n                        help=\"Pass trust_remote_code=True (unnecessary for Mistral)\")\n    parser.add_argument(\"--max_new_tokens\", type=int, default=1280,\n                        help=\"Tokens to generate in the post-training self-test\")\n    parser.add_argument(\"--merge\", action=\"store_true\",\n                        help=\"Also save a merged fp16 model (needs ~14-16GB)\")\n    parser.add_argument(\"--check_data\", action=\"store_true\",\n                        help=\"Offline data + prompt-parity probe (no model load)\")\n    parser.add_argument(\"--smoke\", action=\"store_true\",\n                        help=\"3-step CUDA-guarded end-to-end run (no-op without a GPU)\")\n\n    args = parser.parse_args()\n    args.target_modules = parse_target_modules(args.target_modules)\n\n    # Offline probes first (these never load the 7B model unless a GPU is present).\n    if args.check_data:\n        sys.exit(run_check_data(args))\n    if args.smoke:\n        sys.exit(run_smoke(args))\n\n    # ── Real training path ──\n    check_imports()                      # friendly message if the ML stack is absent\n    import torch\n    if not torch.cuda.is_available():\n        print(\"[!] WARNING: No GPU detected.\")\n        print(\"    QLoRA fine-tuning needs a GPU with >=16GB VRAM (Colab T4 / Kaggle).\")\n        print(\"    Continuing anyway — will be extremely slow on CPU.\")\n        print()\n\n    print(\"=\" * 60)\n    print(\"  QLoRA Fine-Tuning — Cyber Attack Scenario Generation\")\n    print(\"=\" * 60)\n    print(f\"  Base model : {args.base}\")\n    print(f\"  Dataset    : {args.data}\")\n    print(f\"  Output     : {args.out}\")\n    print(\"=\" * 60)\n    print()\n\n    if not Path(args.data).exists():\n        print(f\"[!] Dataset not found: {args.data}\")\n        print(\"    Run generate_outputs.py first.\")\n        sys.exit(1)\n\n    Path(args.out).mkdir(parents=True, exist_ok=True)\n    train(args)\n\n\nif __name__ == \"__main__\":\n    main()\n", encoding='utf-8')
pathlib.Path("fine_tune.py").write_text("\"\"\"Compatibility launcher for the QLoRA fine-tuning script.\n\nThis file lets Colab and local commands run:\n    python fine_tune.py --data training_data.jsonl --out ./output\n\"\"\"\n\nfrom finetuning import main\n\n\nif __name__ == \"__main__\":\n    main()\n", encoding='utf-8')
pathlib.Path("requirements_finetune.txt").write_text("# Fine-tuning dependencies\n# Install with: pip install -r fine_tuning/requirements_finetune.txt\n#\n# Pinned to the stack that produced the shipped adapter (see output/README.md):\n#   Transformers 5.x, TRL 0.29.x, PEFT 0.18.x, datasets 4.x, torch 2.10, bnb 0.49.x.\n# The code branches on the TRL/transformers API differences (max_length vs\n# max_seq_length, processing_class vs tokenizer, the completion-only collator),\n# so nearby versions also work; these ranges are the verified-compatible set.\n\ntorch>=2.4\ntransformers>=5.0,<6\ntrl>=0.29,<0.30\npeft>=0.18,<0.19\ndatasets>=4.0,<5\naccelerate>=1.1\nbitsandbytes>=0.47,<0.50\n", encoding='utf-8')
print('Wrote:', "prompt_format.py", "data_split.py", "finetuning.py", "fine_tune.py", "requirements_finetune.txt")

In [ ]:
from google.colab import files
print('Upload training_data.jsonl')
up = files.upload()
import os; print('cwd files:', os.listdir('.'))

In [ ]:
# Runs the fixed QLoRA pipeline. The 17 held-out records listed in
# data_split.py are never trained on, so evaluation stays honest.
!python fine_tune.py --data training_data.jsonl --out ./output
# Override examples:
#   --target_modules attn   (attention-only LoRA)
#   --epochs 5 --lr 1e-4 --r 32 --alpha 64
#   --merge                 (also save a merged fp16 model; needs ~16GB)

In [ ]:
import shutil
from google.colab import files
shutil.make_archive('finetuned_adapters', 'zip', './output')
files.download('finetuned_adapters.zip')